# Quantum Circuit Smell Intelligence — Kaggle run

Runs `qcs_pipeline` (the rule-mining "smell detector" pipeline) directly against
the `veerukhannan/mnisq-optbench-pairs` dataset's pre-computed `(input_qasm, target_qasm)`
pairs — Qiskit's transpiler has already been run to produce these, so Step 1 here
just re-diffs the already-paired QASM instead of re-transpiling from scratch.

**Before running:**
1. Attach the dataset: **Add Input** (right sidebar) → search `mnisq-optbench-pairs` → Add.
2. Turn on **Internet**: Notebook settings (right sidebar) → Internet → On. Needed for `pip install`.

The source repo (`veerakrish/quantum-circuit-smell-intelligence`) is **public**, so no
token or Kaggle Secret is needed — plain `pip install git+https://...` works in both
interactive and automated ("Save & Run All" / papermill) execution.

**Run cells in order, top to bottom, in one kernel session.** Restarting the kernel
clears installed packages, not just variables — rerun from the top if you do.

In [ ]:
# Confirm what's actually mounted under /kaggle/input — the exact path can
# differ by execution mode (interactive vs. "Save & Run All"), which is why
# the dataset-locating cell below searches broadly rather than assuming a
# fixed path.
import glob
print(glob.glob('/kaggle/input/*'))
print(glob.glob('/kaggle/input/**/*', recursive=True)[:20])

In [ ]:
# Public repo -> no token, no Secrets setup needed.
!pip install -q "quantum-circuit-smell-intelligence[mining] @ git+https://github.com/veerakrish/quantum-circuit-smell-intelligence.git"

In [ ]:
# Verify the install actually worked before going any further — fail loudly
# here with a clear reason, instead of a bare ModuleNotFoundError several
# cells later with no context.
try:
    import qcs_pipeline
    print("qcs_pipeline imported OK from:", qcs_pipeline.__file__)
except ModuleNotFoundError as e:
    raise RuntimeError(
        "qcs_pipeline is not importable. Scroll up to the pip install cell's "
        "output for the real error (most likely cause now: Internet is off "
        "in notebook settings, or the pip install cell above hasn't been run "
        "yet in this kernel session)."
    ) from e

In [ ]:
# Locate the pairs_k*.parquet chunk files — search all of /kaggle/input rather
# than assuming the dataset lands at /kaggle/input/mnisq-optbench-pairs;
# the mount path has been observed to vary (e.g. /kaggle/input/datasets/...)
# depending on execution mode.
from pathlib import Path
from qcs_pipeline.mining.from_kaggle_pairs import find_pair_chunks

chunk_files = find_pair_chunks(Path("/kaggle/input"))
DATASET_ROOT = chunk_files[0].parent  # the directory that actually held them, whatever it's named
print(f"Found {len(chunk_files)} chunk files under {DATASET_ROOT}")
print(chunk_files[:5])

## Inspect the original pipeline source (ground truth for column meanings)

The dataset bundles the code that generated it, under a sibling `mnisq-optbench/`
directory (visible in the recursive listing two cells up). Read `verify.py` and
`transpile_pairs.py` directly rather than guessing what `fidelity_input`,
`fidelity_target`, `opt_level`, and `transpiler_seed` mean — this is exactly the
check the design discussion flagged as outstanding before trusting the columns.

In [ ]:
# Locate the bundled source tree (search broadly — same reasoning as the
# dataset-chunk discovery above) and print the files most likely to define
# the ambiguous columns.
import glob
from pathlib import Path

SOURCE_FILES_OF_INTEREST = [
    "verify.py",          # almost certainly computes fidelity_input / fidelity_target
    "transpile_pairs.py", # almost certainly generates input_qasm / target_qasm / opt_level / transpiler_seed
    "zx_opt.py",          # relevant if the "optimizer" column includes a PyZX path alongside Qiskit
]

for filename in SOURCE_FILES_OF_INTEREST:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        print(f"=== {filename}: NOT FOUND ===\n")
        continue
    path = Path(matches[0])
    print(f"=== {path} ===")
    print(path.read_text())
    print("\n" + "=" * 80 + "\n")

## Peek before committing

Check the actual `opt_level` distribution and what `fidelity_input`/`fidelity_target`
hold, per the caveats from the design discussion — don't assume, verify.

In [ ]:
import pandas as pd

df_peek = pd.read_parquet(chunk_files[0])
print(df_peek.columns.tolist())
print("\nopt_level value counts:")
print(df_peek["opt_level"].value_counts())
print("\nfidelity_input / fidelity_target sample:")
print(df_peek[["fidelity_input", "fidelity_target", "input_n_gates", "target_n_gates"]].head())

In [ ]:
# How many CPU cores are actually available? Don't just assume 5 -- Kaggle's
# standard CPU notebook allocation has historically been 4 cores (check the
# printed number below for THIS session, it can change). mine_from_parquet
# caps n_workers to this automatically if you ask for more, but it's worth
# knowing up front rather than being surprised by the cap.
import os
print("CPUs available in this session:", os.cpu_count())

In [ ]:
# Step 1 — mine patterns from every chunk (fast path: reuses the already-
# transpiled target_qasm, no re-transpiling). Set opt_level_filter to whatever
# the cell above shows is the dominant/consistent level. Start with
# max_rows_per_chunk set low to sanity-check the run within a few minutes,
# then rerun with None for the full pass.
#
# n_workers splits the 120 chunk files round-robin across processes -- mining
# does no quantum simulation (pure parsing + diffing), so this is safe,
# straightforward CPU parallelism. Set to os.cpu_count() from the cell above
# rather than a fixed number; requesting more than that just oversubscribes
# and runs slower, which mine_from_parquet will warn about and cap for you.
from qcs_pipeline.mining.from_kaggle_pairs import mine_from_parquet

mine_from_parquet(
    dataset_root=DATASET_ROOT,
    out_path=Path("/kaggle/working/mined_pairs.jsonl"),
    opt_level_filter=3,       # match phase0/horizontal_pairs.py's default optimization_level=3 — adjust per the cell above
    max_rows_per_chunk=500,   # sanity-check pass; set to None for the full dataset
    n_workers=os.cpu_count(), # parallelize across all available cores
)

In [ ]:
# Step 2 — canonicalize into a deduped, frequency-ranked rule database
from qcs_pipeline.rules.rule_database import build_rule_database

db = build_rule_database(Path("/kaggle/working/mined_pairs.jsonl"), min_frequency=2)
db.to_json(Path("/kaggle/working/rules.json"))

entries = db.entries()
print(f"{len(entries)} unique rules ({sum(e.conflict for e in entries)} flagged conflicting)")
for e in sorted(entries, key=lambda x: -x.frequency)[:10]:
    print(e.frequency, [op.name for op in e.pattern], "->", [op.name for op in e.rewrite])

In [ ]:
# Apply the smell detector + exact-fidelity verified repair to a sample circuit
from qcs_pipeline.pipeline import QuantumCircuitSmellOptimizer

optimizer = QuantumCircuitSmellOptimizer(rule_db_path=Path("/kaggle/working/rules.json"))

sample_raw_qasm = df_peek.iloc[0]["input_qasm"]
result = optimizer.optimize(sample_raw_qasm)

print(f"{result.n_gates_before} -> {result.n_gates_after} gates")
print("fidelity:", result.fidelity)
print("rule applications:", len(result.applied_smells))
print("quarantined this run:", result.quarantined_rule_count)

## Notes

- **Memory:** with `n_workers` processes active, peak RAM is roughly
  `n_workers` × one parquet chunk (~150 MB each) at once — e.g. ~600 MB for
  4 workers — plus whatever a single circuit's exact-fidelity simulation
  needs during the later `optimize()` call. See the RAM discussion for how
  that scales with qubit count.
- **Parallelism ceiling:** more workers than `os.cpu_count()` doesn't help —
  `mine_from_parquet` caps and warns automatically, but it's worth checking
  the actual core count for this session rather than assuming a number.
- **Session limits:** Kaggle notebook sessions are time-boxed (check the current
  published limit in Kaggle's docs — it has changed over time). For a full-dataset
  run, set `max_rows_per_chunk=None` and consider running it as a saved/scheduled
  Kaggle notebook version rather than interactively.
- **Outputs:** `/kaggle/working/` persists as the notebook's output — `mined_pairs.jsonl`
  and `rules.json` will be downloadable from the notebook's Output tab after a run.
- If `opt_level` turns out to be mixed rather than a single dominant value, rerun
  Step 1 once per level and compare the resulting rule databases rather than merging.